In [ ]:
import os
import zipfile
import numpy as np
import nrrd
from skimage import measure, morphology
from scipy import ndimage
import trimesh

# =========================================================
# ⚙️ CONFIGURACIÓN — EDITA AQUÍ
# =========================================================
ZIPS = {
    "unet_sin_da":    r"E:\PRE\UNET.zip",
    "unet_con_da":    r"E:\PRE\UNETDATA.zip",
    "resunet_sin_da": r"E:\PRE\RES.zip",
    "resunet_con_da": r"E:\PRE\RESDATA.zip",
}
OUT_DIR = r"D:\TFG1"
PACIENTES_INTERES = ["Paciente32", "Paciente60", "Paciente47"]
LABELS_TO_PROCESS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

# =========================================================
# 🗂️ NOMBRES Y CONFIGURACIÓN POR CLASE
# =========================================================
LABEL_TO_NAME = {
    1: "rinones", 2: "higado", 3: "estomago", 4: "pancreas",
    5: "pulmon_izq", 6: "pulmon_dcho", 7: "esofago", 8: "traquea",
    9: "tiroides", 10: "huesos", 11: "corazon", 12: "sangre",
    13: "medula_espinal", 14: "musculo", 15: "piel",
}

LABEL_CONFIG = {
    1:  dict(min_size=200,  keep_largest_cc=False, fill_holes=True,  closing_radius=1, smooth_iterations=8,  step_size=1),
    2:  dict(min_size=1000, keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=10, step_size=1),
    3:  dict(min_size=500,  keep_largest_cc=True,  fill_holes=True,  closing_radius=2, smooth_iterations=10, step_size=1),
    4:  dict(min_size=100,  keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=8,  step_size=1),
    5:  dict(min_size=1000, keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=10, step_size=1),
    6:  dict(min_size=1000, keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=10, step_size=1),
    7:  dict(min_size=50,   keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=5,  step_size=1),
    8:  dict(min_size=50,   keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=5,  step_size=1),
    9:  dict(min_size=50,   keep_largest_cc=False, fill_holes=True,  closing_radius=1, smooth_iterations=5,  step_size=1),
    10: dict(min_size=50,   keep_largest_cc=False, fill_holes=True,  closing_radius=0, smooth_iterations=3,  step_size=1),
    11: dict(min_size=1000, keep_largest_cc=True,  fill_holes=True,  closing_radius=2, smooth_iterations=15, step_size=1),
    12: dict(min_size=200,  keep_largest_cc=False, fill_holes=True,  closing_radius=1, smooth_iterations=8,  step_size=1),
    13: dict(min_size=50,   keep_largest_cc=True,  fill_holes=True,  closing_radius=1, smooth_iterations=5,  step_size=1),
    14: dict(min_size=500,  keep_largest_cc=False, fill_holes=True,  closing_radius=1, smooth_iterations=8,  step_size=1,  process=True),
    15: dict(min_size=500,  keep_largest_cc=False, fill_holes=True,  closing_radius=1, smooth_iterations=10, step_size=1,  process=False, fill_first=True),
}

# =========================================================
# 📂 CARGA VOLUMEN
# =========================================================
def load_nrrd_volume(nrrd_path):
    vol, header = nrrd.read(nrrd_path)
    vol = np.asarray(vol)
    spacing = (2.5,2.5,2.5)
    if "space directions" in header:
        sd = header["space directions"]
        try:
            norms = [float(np.linalg.norm(v)) for v in sd if v is not None and np.linalg.norm(v) > 0]
            if len(norms) == 3:
                spacing = tuple(norms)
        except Exception:
            pass
    return vol, spacing

# =========================================================
# 🧹 POSTPROCESADO
# =========================================================
def postprocess_mask(mask, min_size, keep_largest_cc, fill_holes, closing_radius, fill_first=False):
    mask = mask.astype(bool)
    if fill_first:
        if fill_holes:
            mask = ndimage.binary_fill_holes(mask)
        if closing_radius > 0:
            mask = morphology.binary_closing(mask, footprint=morphology.ball(closing_radius))
    else:
        if closing_radius > 0:
            mask = morphology.binary_closing(mask, footprint=morphology.ball(closing_radius))
        if fill_holes:
            mask = ndimage.binary_fill_holes(mask)
    if min_size > 0:
        mask = morphology.remove_small_objects(mask, min_size=min_size)
    if keep_largest_cc:
        labeled, num = ndimage.label(mask)
        if num > 0:
            counts = ndimage.sum(mask, labeled, index=np.arange(1, num + 1))
            largest = int(np.argmax(counts)) + 1
            mask = (labeled == largest)
    return mask.astype(np.uint8)

# =========================================================
# 🧱 MARCHING CUBES + SUAVIZADO
# =========================================================
def mask_to_mesh(mask, spacing, step_size, smooth_iterations, process=True):
    if np.sum(mask) == 0:
        raise ValueError("Máscara vacía.")
    verts, faces, _, _ = measure.marching_cubes(
        volume=mask.astype(np.float32), level=0.5,
        spacing=spacing, step_size=step_size
    )
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=process)
    if smooth_iterations > 0:
        trimesh.smoothing.filter_laplacian(mesh, iterations=smooth_iterations)
    return mesh

# =========================================================
# 🎯 GENERAR MALLAS DE UN PACIENTE
# =========================================================
def generate_meshes_for_patient(nrrd_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    vol, spacing = load_nrrd_volume(nrrd_path)
    print(f"  Volumen: {vol.shape} | Spacing: {spacing}")

    for label in LABELS_TO_PROCESS:
        name = LABEL_TO_NAME.get(label, f"clase_{label}")
        config = LABEL_CONFIG[label]
        print(f"  [{label:02d}] {name} ...", end=" ", flush=True)
        mask = (vol == label).astype(np.uint8)
        if mask.sum() == 0:
            print("⚠️  Sin voxels.")
            continue
        try:
            mask_pp = postprocess_mask(
                mask=mask,
                min_size=config["min_size"],
                keep_largest_cc=config["keep_largest_cc"],
                fill_holes=config["fill_holes"],
                closing_radius=config["closing_radius"],
                fill_first=config.get("fill_first", False),
            )
            if mask_pp.sum() == 0:
                print("⚠️  Vacía tras postprocesado.")
                continue
            mesh = mask_to_mesh(
                mask_pp, spacing,
                config["step_size"],
                config["smooth_iterations"],
                config.get("process", True),
            )
            out_path = os.path.join(out_dir, f"{label:02d}_{name}.stl")
            mesh.export(out_path)
            print(f"✅  verts={len(mesh.vertices):,} | faces={len(mesh.faces):,}")
        except Exception as e:
            print(f"❌  {e}")

# =========================================================
# ▶️ EJECUCIÓN
# =========================================================
if __name__ == "__main__":
    for model_name, zip_path in ZIPS.items():
        print(f"\n{'='*60}")
        print(f"Modelo: {model_name}")
        print(f"{'='*60}")
        with zipfile.ZipFile(zip_path, 'r') as z:
            for file in z.namelist():
                if not file.endswith("_preds_volume.nrrd"):
                    continue
                for paciente in PACIENTES_INTERES:
                    if paciente not in file:
                        continue
                    print(f"\nProcesando {paciente}...")
                    tmp_path = os.path.join(OUT_DIR, "tmp.nrrd")
                    os.makedirs(OUT_DIR, exist_ok=True)
                    with z.open(file) as src, open(tmp_path, 'wb') as dst:
                        dst.write(src.read())
                    out_dir = os.path.join(OUT_DIR, model_name, paciente)
                    generate_meshes_for_patient(tmp_path, out_dir)
                    os.remove(tmp_path)

    print("\n✅ Proceso completado.")